In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Notebook Magic Command für inline Plots
%matplotlib inline

# Standardgröße für unsere Plots (Breite, Höhe) - ideal für 3 Bilder nebeneinander
plt.rcParams['figure.figsize'] = (18, 6)

Reinhard Logik

In [ ]:
def get_mean_std(image):
    mean, std = cv2.meanStdDev(image)
    return mean.flatten(), std.flatten()

def normalize_stain_reinhard(src_img, target_img):
    # Konvertierung zu LAB & float32
    src_lab = cv2.cvtColor(src_img, cv2.COLOR_BGR2LAB).astype(np.float32)
    target_lab = cv2.cvtColor(target_img, cv2.COLOR_BGR2LAB).astype(np.float32)

    # Statistiken
    src_mean, src_std = get_mean_std(src_lab)
    target_mean, target_std = get_mean_std(target_lab)

    # Kanäle splitten
    l, a, b = cv2.split(src_lab)

    # Zero-Division Schutz
    src_std[src_std == 0] = 1e-5

    # Reinhard Gleichung
    l_norm = (l - src_mean[0]) * (target_std[0] / src_std[0]) + target_mean[0]
    a_norm = (a - src_mean[1]) * (target_std[1] / src_std[1]) + target_mean[1]
    b_norm = (b - src_mean[2]) * (target_std[2] / src_std[2]) + target_mean[2]

    # Merge, Clip und zurück zu BGR (8-Bit)
    result_lab = cv2.merge((l_norm, a_norm, b_norm))
    result_lab = np.clip(result_lab, 0, 255).astype(np.uint8)
    result_bgr = cv2.cvtColor(result_lab, cv2.COLOR_LAB2BGR)

    return result_bgr

In [ ]:
# Hilfsfunktion für die korrekte Farbdarstellung im Plot
def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 1. Bilder laden (Pfade anpassen, falls nötig! '../' geht eine Ebene hoch aus dem notebook Ordner)
src_path = '../data/raw/source.tif'
target_path = '../data/raw/target.tif'

src_img = cv2.imread(src_path)
target_img = cv2.imread(target_path)

# Kurzer Check, ob die Bilder gefunden wurden
if src_img is None or target_img is None:
    print("Fehler: Bilder konnten nicht geladen werden. Bitte Pfade prüfen!")
else:
    # 2. Die Normalisierung anwenden (wie das Anwenden einer dynamischen LUT)
    result_img = normalize_stain_reinhard(src_img, target_img)

    # 3. Visualisierung (Das Portfolio-Layout)
    fig, axes = plt.subplots(1, 3)
    
    axes[0].imshow(bgr_to_rgb(src_img))
    axes[0].set_title('1. Source (Original)')
    axes[0].axis('off') # Versteckt die Pixel-Achsen für einen cleanen Look
    
    axes[1].imshow(bgr_to_rgb(target_img))
    axes[1].set_title('2. Target (Reference Style)')
    axes[1].axis('off')
    
    axes[2].imshow(bgr_to_rgb(result_img))
    axes[2].set_title('3. Result (Normalized)')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()